In [ ]:
!pip install langchain langchain-core langchain-community langchain-text-splitters langchain-huggingface faiss-cpu pypdf sentence-transformers huggingface_hub streamlit

In [3]:
import langchain
import faiss
import pypdf
from pathlib import Path

print("Libraries OK")
print(list(Path("data/documents").glob("*.pdf")))

Libraries OK
[WindowsPath('data/documents/Nova_Tech_Employee_Handbook.pdf')]


In [4]:
import os
from getpass import getpass

token = getpass("Paste your Hugging Face token: ")
os.environ["HF_TOKEN"] = token                      
os.environ["HUGGINGFACEHUB_API_TOKEN"] = token      

from huggingface_hub import whoami
print(whoami()["name"])

Paste your Hugging Face token:  ········


Sameer030


In [22]:
DOCS_DIR = "data/documents"
INDEX_DIR = "faiss_index"

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
LLM_REPO_ID = "meta-llama/Llama-3.1-8B-Instruct"

CHUNK_SIZE = 800
CHUNK_OVERLAP = 150
TOP_K = 4

In [23]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader

docs = []
for pdf_path in Path(DOCS_DIR).glob("*.pdf"):
    loader = PyPDFLoader(str(pdf_path))
    docs.extend(loader.load())

print("Pages loaded:", len(docs))
print(docs[0].metadata)
print(docs[0].page_content[:500])

Pages loaded: 6
{'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-10-01T15:21:57+00:00', 'author': 'Nova Tech Solutions (sample document)', 'keywords': '', 'moddate': '2026-10-01T15:21:57+00:00', 'subject': '(unspecified)', 'title': 'Nova Tech Solutions - Employee Handbook', 'trapped': '/False', 'source': 'data\\documents\\Nova_Tech_Employee_Handbook.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1'}
Nova Tech Solutions
Employee Handbook 2026
Sample document created for a RAG chatbot practice project. Company and policies are
fictional.
1. About the Company
Nova Tech Solutions is a software services company founded in 2015 and headquartered
in Hyderabad, India. The company has about 450 employees and additional offices in
Bengaluru and Pune. Nova Tech builds web applications, mobile apps and data platforms
for clients in banking, healthcare and retail.
Our core values are Integrity, Ownershi


In [24]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)
chunks = splitter.split_documents(docs)

print("Total chunks:", len(chunks))
print(chunks[5].page_content)
print(chunks[5].metadata)

Total chunks: 13
reimbursement of 1,000 INR is also provided.
5. Expense Reimbursement and Travel
Employees can claim reimbursement for business expenses such as client travel,
accommodation, meals during travel and approved training courses. All claims must be
submitted through the NovaHR expense module within 30 days of the expense.
Original receipts are required for any single expense above 500 INR. Claims without
receipts above this amount will be rejected.
Domestic air travel must be booked in economy class. Hotel stays are limited to 6,000 INR
per night in metro cities and 4,000 INR per night in other cities. The daily meal allowance
during travel is 800 INR.
Reimbursements are approved by the reporting manager first and then by the Finance
{'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-10-01T15:21:57+00:00', 'author': 'Nova Tech Solutions (sample document)', 'keywords': '', 'moddate': '2026-10-01T15:21:57+00:00', 'subject':

In [8]:
import torch
print(torch.__version__)

2.14.1+cpu


In [25]:
import os
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

In [26]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)
vectorstore = FAISS.from_documents(chunks, embeddings)
print("Index built with", vectorstore.index.ntotal, "vectors")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Index built with 13 vectors


In [27]:
vectorstore.save_local(INDEX_DIR)
print("Saved to", INDEX_DIR)

Saved to faiss_index


In [28]:
results = vectorstore.similarity_search("How many sick leave days do I get?", k=1)
for i, doc in enumerate(results, 1):
    print(f"\n--- Chunk {i} | page {doc.metadata['page'] + 1} ---")
    print(doc.page_content[:300])


--- Chunk 1 | page 1 ---
Attendance is recorded using the biometric system in the office and the NovaHR mobile
app for remote days.
3. Leave Policy
Annual leave: Every full-time employee gets 18 paid leave days per year, credited at the
start of January. Of these, 12 are casual leave days and 6 are sick leave days.
Sick lea


In [29]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)

vectorstore = FAISS.load_local(
    INDEX_DIR,
    embeddings,
    allow_dangerous_deserialization=True,
)
print("Loaded vectors:", vectorstore.index.ntotal)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loaded vectors: 13


In [30]:
retriever = vectorstore.as_retriever(search_kwargs={"k": TOP_K})

In [31]:
def show(question):
    results = retriever.invoke(question)
    print("QUESTION:", question)
    for i, doc in enumerate(results, 1):
        page = doc.metadata["page"] + 1
        text = doc.page_content[:250].replace("\n", " ")
        print(f"  [{i}] page {page}: {text}")
    print()

show("How many sick leave days do I get?")

QUESTION: How many sick leave days do I get?
  [1] page 1: Attendance is recorded using the biometric system in the office and the NovaHR mobile app for remote days. 3. Leave Policy Annual leave: Every full-time employee gets 18 paid leave days per year, credited at the start of January. Of these, 12 are cas
  [2] page 2: Maternity leave: Female employees are entitled to 26 weeks of paid maternity leave. Paternity leave: Male employees are entitled to 10 working days of paid paternity leave, to be taken within 3 months of the child's birth. Public holidays: The compan
  [3] page 1: question not covered here, employees should contact the HR team at hr@novatech-example.com or dial extension 200. 2. Working Hours and Attendance The standard working week is Monday to Friday. Office hours are 9:30 AM to 6:30 PM, which includes a one
  [4] page 6: Referral bonus: Employees receive 30,000 INR for referring a candidate who is hired and completes 6 months with the company. Free lunch is provide

In [32]:
questions = [
    "What is the notice period after confirmation?",
    "How much is the hotel limit per night in metro cities?",
    "How many days of paternity leave are there?",
    "What is the password policy?",
    "How much is the learning budget?",
    "Can I do freelance work?",
    "What is the CEO's name?",
]

for q in questions:
    show(q)

QUESTION: What is the notice period after confirmation?
  [1] page 6: Referral bonus: Employees receive 30,000 INR for referring a candidate who is hired and completes 6 months with the company. Free lunch is provided on all office days at the cafeteria in the Hyderabad and Bengaluru offices. 10. Joining, Notice Period
  [2] page 1: question not covered here, employees should contact the HR team at hr@novatech-example.com or dial extension 200. 2. Working Hours and Attendance The standard working week is Monday to Friday. Office hours are 9:30 AM to 6:30 PM, which includes a one
  [3] page 1: Attendance is recorded using the biometric system in the office and the NovaHR mobile app for remote days. 3. Leave Policy Annual leave: Every full-time employee gets 18 paid leave days per year, credited at the start of January. Of these, 12 are cas
  [4] page 2: Maternity leave: Female employees are entitled to 26 weeks of paid maternity leave. Paternity leave: Male employees are entitled to 10 

In [17]:
import re

tests = {
    "How many sick leave days do I get?": "sick leave days",
    "What is the notice period after confirmation?": "60 days",
    "How much is the hotel limit per night in metro cities?": "6,000 INR per night",
    "How many days of paternity leave are there?": "10 working days",
    "What is the password policy?": "12 characters",
    "How much is the learning budget?": "25,000",
    "Can I do freelance work?": "freelance",
}

def norm(text):
    return re.sub(r"\s+", " ", text).lower()

def check(retriever, tests):
    found = 0
    for question, keyword in tests.items():
        docs = retriever.invoke(question)
        rank = next((i for i, d in enumerate(docs, 1)
                     if keyword.lower() in norm(d.page_content)), None)
        print(f"rank {rank}" if rank else "MISS  ", "|", question)
        found += rank is not None
    print(f"\n{found}/{len(tests)} questions have the answer in the top {len(docs)}")

check(retriever, tests)

rank 1 | How many sick leave days do I get?
rank 1 | What is the notice period after confirmation?
rank 1 | How much is the hotel limit per night in metro cities?
rank 1 | How many days of paternity leave are there?
rank 1 | What is the password policy?
rank 1 | How much is the learning budget?
rank 1 | Can I do freelance work?

7/7 questions have the answer in the top 4


In [33]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

for size, overlap in [(400, 80), (800, 150), (1200, 200)]:
    splitter = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=overlap)
    test_chunks = splitter.split_documents(docs)
    test_store = FAISS.from_documents(test_chunks, embeddings)
    for k in [2, 4]:
        r = test_store.as_retriever(search_kwargs={"k": k})
        hits = sum(
            any(kw.lower() in norm(d.page_content) for d in r.invoke(q))
            for q, kw in tests.items()
        )
        print(f"chunk_size={size:>4}  k={k}  chunks={len(test_chunks):>3}  score={hits}/{len(tests)}")

chunk_size= 400  k=2  chunks= 25  score=7/7
chunk_size= 400  k=4  chunks= 25  score=7/7
chunk_size= 800  k=2  chunks= 13  score=7/7
chunk_size= 800  k=4  chunks= 13  score=7/7
chunk_size=1200  k=2  chunks= 11  score=6/7
chunk_size=1200  k=4  chunks= 11  score=7/7


In [34]:
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace

endpoint = HuggingFaceEndpoint(
    repo_id=LLM_REPO_ID,
    task="text-generation",
    max_new_tokens=512,
    temperature=0.2,
)
llm = ChatHuggingFace(llm=endpoint)

In [35]:
response = llm.invoke("Say hello in one short sentence.")
print(response.content)

Hello!


In [36]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a helpful HR assistant for Nova Tech Solutions. "
     "Answer the question using ONLY the context below. "
     "If the answer is not in the context, say: \"I couldn't find that in the handbook.\" "
     "Do not make things up.\n\nContext:\n{context}"),
    ("human", "{question}"),
])

In [37]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [38]:
questions = [
    "How many sick leave days do I get?",
    "What is the notice period after confirmation?",
    "What is the CEO's name?",
]

for q in questions:
    print("Q:", q)
    print("A:", rag_chain.invoke(q))
    print()

Q: How many sick leave days do I get?
A: You get 6 sick leave days per year.

Q: What is the notice period after confirmation?
A: After confirmation, the notice period is 60 days for all employees up to Senior Engineer level and 90 days for Lead and above.

Q: What is the CEO's name?
A: I couldn't find that in the handbook.



In [43]:
%%writefile ChatBot.py
import os
import time
from pathlib import Path

import streamlit as st
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_huggingface import (
    ChatHuggingFace,
    HuggingFaceEmbeddings,
    HuggingFaceEndpoint,
)

# ---------------------------------------------------------------- settings
INDEX_DIR = "faiss_index"
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
LLM_REPO_ID = "meta-llama/Llama-3.1-8B-Instruct"
TOP_K = 4
CHUNK_SIZE = 800
CHUNK_OVERLAP = 150

SAMPLE_QUESTIONS = [
    "How many sick leave days do I get?",
    "What is the notice period after confirmation?",
    "What is the hotel limit per night in metro cities?",
    "What is the password policy?",
    "How much is the learning budget?",
]

st.set_page_config(page_title="Nova Tech HR Assistant", layout="centered")
st.title("Nova Tech HR Assistant")
st.caption(
    "Answers grounded in the Nova Tech Employee Handbook 2026, "
    "with the supporting sources shown for every answer."
)

# ---------------------------------------------------------------- token
token = os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACEHUB_API_TOKEN")
if not token:
    token = st.sidebar.text_input("Hugging Face token", type="password")
if not token:
    st.info("Enter your Hugging Face token in the sidebar to start.")
    st.stop()
os.environ["HF_TOKEN"] = token
os.environ["HUGGINGFACEHUB_API_TOKEN"] = token


# ---------------------------------------------------------------- pipeline
@st.cache_resource(show_spinner="Loading models and document index...")
def load_components():
    embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)
    vectorstore = FAISS.load_local(
        INDEX_DIR, embeddings, allow_dangerous_deserialization=True
    )
    retriever = vectorstore.as_retriever(search_kwargs={"k": TOP_K})

    endpoint = HuggingFaceEndpoint(
        repo_id=LLM_REPO_ID,
        task="text-generation",
        max_new_tokens=512,
        temperature=0.2,
    )
    llm = ChatHuggingFace(llm=endpoint)

    prompt = ChatPromptTemplate.from_messages([
        ("system",
         "You are a helpful HR assistant for Nova Tech Solutions. "
         "Answer the question using ONLY the context below. "
         "If the answer is not in the context, say: "
         "\"I couldn't find that in the handbook.\" "
         "Do not make things up.\n\nContext:\n{context}"),
        ("human", "{question}"),
    ])
    chain = prompt | llm | StrOutputParser()
    return retriever, chain


retriever, chain = load_components()

# ---------------------------------------------------------------- state
if "messages" not in st.session_state:
    st.session_state.messages = []
if "pending" not in st.session_state:
    st.session_state.pending = None


def set_pending(q):
    st.session_state.pending = q


def clear_chat():
    st.session_state.messages = []


# ---------------------------------------------------------------- sidebar
with st.sidebar:
    st.subheader("Try a question")
    for i, q in enumerate(SAMPLE_QUESTIONS):
        st.button(q, key=f"sample_{i}", on_click=set_pending, args=(q,),
                  use_container_width=True)
    st.button("Clear conversation", key="clear", on_click=clear_chat,
              use_container_width=True)

    st.divider()
    st.subheader("How it works")
    st.markdown(
        "1. The question is converted into an embedding.\n"
        "2. The most similar handbook chunks are retrieved from a FAISS index.\n"
        "3. The chunks and the question are placed in a prompt.\n"
        "4. The LLM answers using only that context.\n"
        "5. The source chunks are shown so the answer can be verified."
    )

    st.divider()
    st.subheader("Configuration")
    st.markdown(
        f"**LLM:** {LLM_REPO_ID.split('/')[-1]}  \n"
        f"**Embeddings:** {EMBEDDING_MODEL.split('/')[-1]}  \n"
        f"**Vector store:** FAISS  \n"
        f"**Chunk size / overlap:** {CHUNK_SIZE} / {CHUNK_OVERLAP}  \n"
        f"**Top-k retrieved:** {TOP_K}"
    )


# ---------------------------------------------------------------- helpers
def render_sources(sources):
    with st.expander(f"Sources ({len(sources)})"):
        for i, s in enumerate(sources, 1):
            with st.container(border=True):
                st.markdown(f"**[{i}] {s['file']}, page {s['page']}**")
                st.caption(s["text"])


def is_refusal(answer):
    return "couldn't find that" in answer.lower()


AVATARS = {"user": ":material/person:", "assistant": ":material/smart_toy:"}

# ---------------------------------------------------------------- chat view
if not st.session_state.messages:
    st.info(
        "Ask about company policy: leave, working hours, remote work, expenses, "
        "IT security, benefits and exit rules are covered. "
        "Pick a sample question from the sidebar or type your own."
    )

for msg in st.session_state.messages:
    with st.chat_message(msg["role"], avatar=AVATARS[msg["role"]]):
        st.markdown(msg["content"])
        if msg.get("sources"):
            render_sources(msg["sources"])
        if msg.get("elapsed"):
            st.caption(f"Answered in {msg['elapsed']:.1f} s")

# ---------------------------------------------------------------- new question
question = st.chat_input("Ask a question about the handbook")
if st.session_state.pending:
    question = st.session_state.pending
    st.session_state.pending = None

if question:
    st.session_state.messages.append({"role": "user", "content": question})
    with st.chat_message("user", avatar=AVATARS["user"]):
        st.markdown(question)

    with st.chat_message("assistant", avatar=AVATARS["assistant"]):
        try:
            start = time.perf_counter()
            with st.spinner("Searching the handbook..."):
                docs = retriever.invoke(question)
                context = "\n\n".join(d.page_content for d in docs)
                answer = chain.invoke({"context": context, "question": question})
            elapsed = time.perf_counter() - start

            sources = []
            if not is_refusal(answer):
                sources = [
                    {
                        "file": Path(d.metadata.get("source", "unknown")).name,
                        "page": d.metadata.get("page", 0) + 1,
                        "text": d.page_content[:300].replace("\n", " ") + "...",
                    }
                    for d in docs
                ]

            st.markdown(answer)
            if sources:
                render_sources(sources)
            st.caption(f"Answered in {elapsed:.1f} s")

            st.session_state.messages.append(
                {"role": "assistant", "content": answer,
                 "sources": sources, "elapsed": elapsed}
            )
        except Exception as e:
            st.error(
                "The language model could not be reached. Check your token and "
                "internet connection, then try again."
            )
            st.caption(f"Details: {str(e)[:200]}")

Overwriting ChatBot.py


In [44]:
%%writefile requirements.txt
langchain
langchain-core
langchain-community
langchain-text-splitters
langchain-huggingface
faiss-cpu
pypdf
sentence-transformers
huggingface_hub
streamlit
pytest

Writing requirements.txt


In [45]:
%%writefile .gitignore
venv/
.env
__pycache__/
.ipynb_checkpoints/

Writing .gitignore


In [51]:
%%writefile .env.example
HF_TOKEN=your_huggingface_token_here

Overwriting .env.example


In [48]:
import os
os.makedirs("tests", exist_ok=True)

In [49]:
%%writefile tests/test_retriever.py
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"


def load_retriever(k=4):
    embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)
    store = FAISS.load_local(
        "faiss_index", embeddings, allow_dangerous_deserialization=True
    )
    return store.as_retriever(search_kwargs={"k": k})


def test_returns_k_chunks():
    results = load_retriever(k=4).invoke("How many sick leave days do I get?")
    assert len(results) == 4


def test_sick_leave_answer_is_retrieved():
    results = load_retriever().invoke("How many sick leave days do I get?")
    text = " ".join(d.page_content for d in results).lower()
    assert "sick leave" in text


def test_notice_period_answer_is_retrieved():
    results = load_retriever().invoke("What is the notice period after confirmation?")
    text = " ".join(d.page_content.replace("\n", " ") for d in results)
    assert "60 days" in text

Writing tests/test_retriever.py


In [54]:
!python -m pytest tests -q

...                                                                      [100%]
============================== warnings summary ===============================
tests\test_retriever.py:1
  C:\Users\samee\Downloads\Rag-ChatBot\tests\test_retriever.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
    from langchain_community.vectorstores import FAISS

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
3 passed, 1 warning in 27.58s


In [56]:
%%writefile README.md
# Nova Tech HR Assistant: Document Q&A Chatbot with RAG

A Retrieval-Augmented Generation (RAG) chatbot that answers questions about the
Nova Tech Employee Handbook, a fictional sample document. Instead of relying on
what the LLM already knows, the system retrieves the relevant parts of the
document and gives them to the LLM as context. Answers are grounded in the
source, and the supporting passages are shown with every answer.

Built with LangChain, FAISS, Hugging Face and Streamlit.

## Demo

![Answer with sources](screenshots/notice_period.png)
![Question not covered by the document](screenshots/not_found.png)

## How it works

```
OFFLINE (run once)                       ONLINE (every question)
PDF -> split into chunks                 User question
    -> embeddings (MiniLM)                   -> embedding
    -> FAISS index (saved to disk)           -> FAISS returns top-k similar chunks
                                             -> prompt (context + question)
                                             -> LLM (Llama 3.1 8B Instruct)
                                             -> answer + sources shown in Streamlit
```

1. **Load:** the PDF is read page by page with `PyPDFLoader`.
2. **Chunk:** pages are split into overlapping chunks of 800 characters
   (150 overlap) with `RecursiveCharacterTextSplitter`.
3. **Embed and index:** each chunk is converted to a 384-dimension vector with
   `all-MiniLM-L6-v2` and stored in a FAISS index.
4. **Retrieve:** the question is embedded and the 4 closest chunks are returned.
5. **Generate:** the chunks and question go into a prompt that tells the LLM to
   answer only from the context and to say so when the answer is not there.
6. **Show sources:** the file name, page number and a preview of each chunk used
   are displayed under the answer.

## Tech stack

| Layer | Tool |
|---|---|
| Language | Python |
| Orchestration | LangChain (prompt template, retriever, chain built with `\|`) |
| Document loading | PyPDFLoader |
| Chunking | RecursiveCharacterTextSplitter |
| Embeddings | sentence-transformers/all-MiniLM-L6-v2 (runs locally) |
| Vector store | FAISS |
| LLM | meta-llama/Llama-3.1-8B-Instruct via Hugging Face Inference Providers |
| Interface | Streamlit |
| Testing | pytest |

## Project structure

```
data/documents/          Source PDF
faiss_index/             Saved vector index (generated, not committed)
rag_chatbot.ipynb        Ingestion, retrieval experiments, RAG chain
ChatBot.py               Streamlit chat interface
tests/test_retriever.py  Retrieval tests
requirements.txt         Dependencies
screenshots/             Images used in this README
```

## Setup

1. **Clone the repository and install dependencies**
```
   pip install -r requirements.txt
```
2. **Create a Hugging Face access token** at
   https://huggingface.co/settings/tokens with permission to make calls to
   Inference Providers. Make sure at least one inference provider that serves the
   model is enabled in your account settings.
3. **Build the index.** Open `rag_chatbot.ipynb`, enter your token when asked,
   and run the cells through "Save the index". This creates the `faiss_index`
   folder.
4. **Set your token** in the terminal you will run the app from (Windows):
```
   set HF_TOKEN=your_token_here
```
5. **Run the app**
```
   python -m streamlit run ChatBot.py
```
   If you skip step 4, the app asks for the token in the sidebar.
6. **Run the tests** (after the index exists)
```
   python -m pytest tests -q
```

The token is never stored in the project. Do not commit it.

## Retrieval experiments

To choose the chunking settings I wrote 7 test questions, each with a key phrase
that a correct chunk must contain. A question counts as retrieved if that phrase
appears in one of the returned chunks.

| Chunk size | Top-k | Chunks | Score |
|---|---|---|---|
| 400 | 2 | 25 | 7/7 |
| 400 | 4 | 25 | 7/7 |
| 800 | 2 | 13 | 7/7 |
| 800 | 4 | 13 | 7/7 |
| 1200 | 2 | 11 | 6/7 |
| 1200 | 4 | 11 | 7/7 |

**Chosen settings: chunk size 800, overlap 150, top-k 4.** It retrieved every
answer and produced readable, self-contained chunks. Larger chunks mix several
topics and dilute the match (1200 with k=2 missed a question). Very small chunks
can split a rule from its context.

Limits of this test: the question set is small, and it measures retrieval, not
the quality of the final generated answer.

## Example questions

| Question | Behaviour |
|---|---|
| How many sick leave days do I get? | Answers 6 and shows the source chunk |
| What is the notice period after confirmation? | 60 days up to Senior Engineer, 90 days for Lead and above |
| What is the hotel limit per night in metro cities? | 6,000 INR |
| What is the CEO's name? | "I couldn't find that in the handbook." (not in the document) |

## Challenges and how I solved them

- **Token not recognised.** `huggingface_hub` reads `HF_TOKEN`, not the variable
  name I first used. I set both and entered the token with `getpass` so it is
  never saved in the notebook.
- **PyTorch failed to load on Windows** (`shm.dll` error). I isolated it by
  testing `import torch` on its own, then repaired the installation.
- **The first LLM was not available.** No enabled provider served
  Qwen2.5-7B-Instruct. I wrote a loop that sent a test message to several
  candidate models, then switched to Llama 3.1 8B Instruct. Because settings live
  in one place, this was a one-line change.
- **The retriever cannot say "I don't know".** Top-k always returns k chunks, even
  when none contain the answer. The prompt therefore instructs the LLM to answer
  only from the context and otherwise say it could not find the answer, and the
  app hides the sources when it declines.

## Limitations and future work

- Each question is answered independently, so there is no conversation memory
  for follow-up questions.
- Only one short document is indexed.
- Answer quality is checked by hand. A larger labelled test set would allow
  measuring it.
- Possible extensions: conversation memory, multiple documents, and a check that
  the answer is supported by the retrieved text before showing it.

## Notes

The handbook is a fictional sample document created for practice. This project
uses LangChain only, without LangGraph.

Writing README.md
